# Using `transformers` Models via Pipelines

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/02_pipelines.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

_Click the badge above to open and run this notebook in Google Colab!_

In [1]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/pipelines/02-pipelines"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Overview

The [`transformers`](https://huggingface.co/docs/transformers/index) library provides models that are faithful to their papers, easy to use, and easy to hack.

- **Engineers** who want a pretrained model that “just works” with a predictable API.
- **Practitioners** fine-tuning, evaluating, or serving models.
- Researchers and educators exploring or extending model architectures (PyTorch-first).

## Set up

To start, we recommend creating a Hugging Face [account](https://hf.co/join). An account lets you host and access version controlled models, datasets, and [Spaces](https://hf.co/spaces) on the Hugging Face [Hub](https://hf.co/docs/hub/index), a collaborative platform for discovery and building.

Create a [User Access Token](https://hf.co/docs/hub/security-tokens#user-access-tokens) and log in to your account.

### Install PyTorch



In [2]:
# !pip install -qqq torch
# !pip install -qqqU transformers datasets evaluate accelerate timm spacy

## Pipeline

The [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline) class is the most convenient way to inference with a pretrained model. It supports many tasks such as text generation, image segmentation, automatic speech recognition, document question answering, and more.

> Refer to the [Pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines) API reference for a complete list of available tasks.

### Different modalities

The `pipeline()` function supports multiple modalities, allowing you to work with text, images, audio, and even multimodal tasks. In this course we’ll focus on text tasks, but it’s useful to understand the transformer architecture’s potential, so we’ll briefly outline it.

| Task | Modality | Output | Models | Datasets |
|---|---|---|---|---|
| [`audio-classification`](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.AudioClassificationPipeline) | Audio | Classify audio into categories | [🔗](https://huggingface.co/models?filter=audio-classification) | [🔗](https://huggingface.co/datasets?task_categories=audio-classification) |
| [`image-classification`](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ImageClassificationPipeline) | Image | Predicted class for an image | [🔗](https://huggingface.co/models?filter=image-classification) | [🔗](https://huggingface.co/datasets?task_categories=image-classification) |
| [`object-detection`](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ObjectDetectionPipeline) | Image | Locate and identify objects in images | [🔗](https://huggingface.co/models?filter=object-detection) | [🔗](https://huggingface.co/datasets?task_categories=object-detection) |

Refer to the [Pipelines](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines) API reference for a complete list of available tasks.


Create a [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline) object and select a task. By default, [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline) downloads and caches a default pretrained model for a given task. Pass the model name to the `model` parameter to choose a specific model.

### Example 1: Text Classification

The simplest classification task: assign a text to one of a **fixed set of labels** the model was trained on. Sentiment analysis is a common case — the model picks either `POSITIVE` or `NEGATIVE` and returns a confidence score.

In [3]:
from transformers import pipeline

sentiment = pipeline(
    task="text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

In [ ]:

sentiment([
    "I love this product!",
    "This is the worst purchase I've ever made.",
])

### Example 2: Zero-shot Classification

That model only knows **POSITIVE** and **NEGATIVE** — labels it was trained on. [*Zero-shot classification*](https://huggingface.co/models?pipeline_tag=zero-shot-classification&sort=trending) goes further: you supply your own candidate labels at inference time, with no retraining.

In [4]:
from transformers import pipeline

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device_map="auto"
)

In [ ]:
sequence_to_classify = "I've been waiting for a HuggingFace course my whole life."
candidate_labels = ['machine learning', 'algebra', 'history']

output = classifier(sequence_to_classify, candidate_labels)
output

### Example 3: Privacy Filter

In [5]:
from transformers import pipeline

token_classifier = pipeline(
    task="token-classification",
    model="openai/privacy-filter",
    aggregation_strategy="simple",  # merge subword tokens into whole entities
)

In [18]:
text = "My name is Alice Smith and my address is 123 Main St, Anytown, USA"
output = token_classifier(text)
# output

The aggregated output is easier to read than raw subword tokens, but a visual highlight makes it clearer still.

In [19]:
from spacy import displacy

displacy.render(
    {"text": text, "ents": [ {"start": ent["start"], "end": ent["end"], "label": ent["entity_group"]} for ent in output ]},
    style="ent",
    manual=True,  # required when passing a dict instead of a spaCy Doc
    jupyter=True,
)

## Batch Inference

Batch inference is disabled by default since  hardware, data, and the model itself can affect whether it improves speed or not. In the example below, when there are 3 inputs and `batch_size=2`, Pipeline passes a batch of 2 inputs, then a batch of 1.

In [4]:
from transformers import pipeline

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device_map="auto",
    batch_size=2,
)

In [7]:
import pandas as pd

df = pd.DataFrame({
    "review_id": [1, 2, 3],
    "text": [
        "I am very happy",
        "The product was great",
        "I'm very sad",
    ],
})

inputs = df["text"]

output = classifier(
    inputs.tolist(),
    candidate_labels=[
        "positive review",
        "negative review",
    ],
)

In [10]:
import pandas as pd

# Create a simplified DataFrame with sequence, top label, and its probability score
df_simplified = pd.DataFrame({
    'sequence': [item['sequence'] for item in output],
    'sentiment': [item['labels'][0] for item in output],
    'probability': [item['scores'][0] for item in output]
})

# Display the updated DataFrame
display(df_simplified)

## Batch Inference: rules of thumb

- If you are latency constrained (live product doing inference), don’t batch.
- The larger the GPU the more likely batching is going to be more interesting.
- If you are using CPU, don’t batch.
- Handle OOM errors.

Read more at: [Pipeline batching](https://huggingface.co/docs/transformers/main_classes/pipelines#pipeline-batching).
